# Imports 

In [14]:
import pandas as pd
from collections import defaultdict
import sys
import os
import shutil as sh
import numpy as np
import seaborn as sns
import os
import importlib
import MDAnalysis as mda
import nglview as nv
from biopandas.pdb import PandasPdb
from io import StringIO
import joblib
from sklearn.metrics import average_precision_score


from sklearn.calibration import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, roc_auc_score

from sklearn.naive_bayes import CategoricalNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial import ConvexHull

from urllib.error import HTTPError
from pathlib import Path
from ipywidgets import interact, interactive, fixed, interact_manual, IntProgress
import ipywidgets as widgets # type: ignore
from IPython.display import display, Markdown, clear_output


# Methods

In [7]:
from sklearn.discriminant_analysis import StandardScaler
from sklearn.feature_selection import r_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import matthews_corrcoef
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder



def load_data(path):
    """
    Load data from the path were it was stored.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"The file at {path} was not found.")
    return pd.read_csv(path)

def preprocess_data(df, columns_to_drop=[], categorical_cols=[]):
    """
    Drop uneccessary columns, 
    Encode categorical features, 
    Handle missing values
    """
    encodings: dict[str, np.ndarray] = {}

    # Drop unwanted columns
    if columns_to_drop:
        drop_list = [c for c in columns_to_drop if c in df.columns]
        df = df.drop(columns=drop_list)

    # Label‐encode only the columns listed in categorical_cols
    if categorical_cols:
        for col in categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                encodings[col] = le.classes_.copy()
                

    # Identify numeric columns and build the pipeline
    num_list = df.select_dtypes(include=[np.number]).columns.tolist()

    num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean'))])
    df[num_list] = num_pipeline.fit_transform(df[num_list])

    return df, encodings

# def preprocess_data(
#     df: pd.DataFrame,
#     y: pd.Series = None,
#     columns_to_drop: list[str]    = None,
#     categorical_cols: list[str]   = None,
#     encoding: str                 = 'label'
# ) -> tuple[pd.DataFrame, dict]:
#     """
#     1) Drop columns in `columns_to_drop`.
#     2) Encode the columns in `categorical_cols` using the chosen strategy:
#          - 'label'  : sklearn LabelEncoder per-column (maps to 0..n-1)
#          - 'onehot' : sklearn OneHotEncoder (expands to dummies; drop='first' to avoid collinearity)
#          - 'target' : category_encoders.TargetEncoder (requires y, mean-target encode)
#     3) Impute missing values in all numeric columns (mean).
    
#     Returns:
#       - df_transformed: DataFrame after drop, encode, impute
#       - encoders:      dict of fitted encoders (or encoder.attrs if multi-col)
#     """
#     df = df.copy()
#     encoders = {}

#     # 1) Drop
#     if columns_to_drop:
#         to_drop = [c for c in columns_to_drop if c in df.columns]
#         df.drop(columns=to_drop, inplace=True)

#     # 2) Encode
#     if categorical_cols:
#         missing = [c for c in categorical_cols if c not in df.columns]
#         if missing:
#             raise ValueError(f"categorical_cols not in df: {missing}")
        
#         if encoding == 'label':
#             # one LabelEncoder per column
#             for col in categorical_cols:
#                 le = LabelEncoder()
#                 df[col] = le.fit_transform(df[col].astype(str))
#                 encoders[col] = le  # you can inspect le.classes_
        
#         elif encoding == 'onehot':
#         # ColumnTransformer to one-hot the listed columns, drop remainder
#             ohe = OneHotEncoder(
#                  handle_unknown='ignore',
#                 drop='first',
#                 sparse_output=False   # use this instead of sparse=False
#             )
#             ct = ColumnTransformer(
#                  [('ohe', ohe, categorical_cols)],
#                 remainder='drop'
#             )
#             arr = ct.fit_transform(df)
#             cols = ct.get_feature_names_out()
#             df = pd.DataFrame(arr, columns=cols, index=df.index)
#             encoders['onehot'] = ct

#     # 3) Impute numeric
#     num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
#     if num_cols:
#         num_pipe = Pipeline([
#             ('imputer', SimpleImputer(strategy='mean'))
#         ])
#         df[num_cols] = num_pipe.fit_transform(df[num_cols])
#         encoders['imputer'] = num_pipe.named_steps['imputer']

#     return df, encoders

def separate_features_target(df, target, columns_to_remove=None):
    if columns_to_remove is None:
        columns_to_remove=[]
    columns_to_remove=set(columns_to_remove + [target])
    X=df.drop(columns=[col for col in columns_to_remove if col in df.columns])
    y=df[target]
    return X, y


def select_features(X, y, threshold=0.1):
    correlations = pd.Series(r_regression(X, y), index=X.columns)
    selected_features = correlations[correlations.abs() >= threshold].index.tolist()
    print(f"The selected features of {X.shape[1]} were: {len(selected_features)}")
    return selected_features, correlations

def train_tuned_model(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    cmap="Blues",
    scale=True
):
    """
    Train and evaluate a tuned model using:
      - Confusion matrix (plotted as heatmap)
      - AUC (if applicable)
      - Matthews Correlation Coefficient (MCC)

    Returns:
      auc (float or None), mcc (float), y_test, y_pred
    """
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual: 0", "Actual: 1"],
        columns=["Predicted: 0", "Predicted: 1"]
    )
    plt.figure(figsize=(5,4))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap=cmap)
    plt.title(f"{model.__class__.__name__} Confusion Matrix")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show()

    auc = None
    if hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X_test)[:,1]
            auc = roc_auc_score(y_test, y_prob)
            print(f"AUC: {auc:.4f}")
        except Exception:
            print("AUC could not be computed from probabilities.")
    else:
        try:
            auc = roc_auc_score(y_test, y_pred)
            print(f"AUC (label‐based): {auc:.4f}")
        except ValueError:
            print("AUC could not be computed (only one class predicted).")

    
    mcc = matthews_corrcoef(y_test, y_pred)
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

    return auc, mcc, y_test, y_pred

# Build Dataset Per Superfamily

In [10]:
df = load_data("/home/user_stel/AISB/Project/dataset/processed_dataset.csv")
# print(df)

In [5]:
groups = { domain_name: sub_df 
           for domain_name, sub_df in df.groupby('domain') }

for domain_name, subset_df in groups.items():
    print(f"Domain: {domain_name}  →  Number of rows: {len(subset_df)}")
    display(subset_df)

Domain: ANNEXIN  →  Number of rows: 9283


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
157547,ANNEXIN,2.0,1.0,55.0,1.0,0.0,0.0,1.0,4.0
157548,ANNEXIN,2.0,0.0,61.0,1.0,0.0,0.0,1.0,4.0
157549,ANNEXIN,2.0,0.0,68.0,1.0,0.0,0.0,1.0,4.0
157550,ANNEXIN,2.0,1.0,17.0,0.0,0.0,0.0,1.0,4.0
157551,ANNEXIN,2.0,1.0,58.0,1.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
166825,ANNEXIN,4.0,0.0,270.0,1.0,1.0,0.0,1.0,1.0
166826,ANNEXIN,4.0,0.0,262.0,1.0,1.0,0.0,1.0,1.0
166827,ANNEXIN,4.0,0.0,243.0,1.0,1.0,0.0,1.0,1.0
166828,ANNEXIN,4.0,0.0,242.0,1.0,1.0,0.0,1.0,1.0


Domain: C1  →  Number of rows: 2088


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
79083,C1,2.0,0.0,156.0,1.0,0.0,0.0,1.0,4.0
79084,C1,2.0,0.0,154.0,0.0,0.0,0.0,1.0,4.0
79085,C1,2.0,0.0,175.0,1.0,0.0,0.0,1.0,4.0
79086,C1,2.0,0.0,199.0,1.0,0.0,0.0,1.0,4.0
79087,C1,2.0,0.0,204.0,1.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
81166,C1,4.0,0.0,79.0,0.0,0.0,0.0,0.0,1.0
81167,C1,4.0,0.0,21.0,1.0,1.0,0.0,1.0,1.0
81168,C1,4.0,1.0,274.0,0.0,0.0,0.0,1.0,1.0
81169,C1,4.0,1.0,268.0,0.0,0.0,0.0,1.0,1.0


Domain: C2  →  Number of rows: 15199


...

,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
139085,PLD,2.0,0.0,319.0,0.0,0.0,0.0,0.0,4.0
139086,PLD,2.0,0.0,324.0,0.0,0.0,0.0,0.0,4.0
139087,PLD,2.0,0.0,328.0,0.0,0.0,0.0,0.0,4.0
139088,PLD,2.0,0.0,409.0,0.0,0.0,0.0,0.0,4.0
139089,PLD,2.0,0.0,462.0,0.0,0.0,0.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...
157542,PLD,4.0,0.0,185.0,0.0,0.0,0.0,0.0,1.0
157543,PLD,4.0,0.0,231.0,0.0,0.0,0.0,0.0,1.0
157544,PLD,4.0,0.0,233.0,0.0,0.0,0.0,0.0,1.0
157545,PLD,4.0,0.0,16.0,0.0,0.0,0.0,0.0,1.0


Domain: PX  →  Number of rows: 5847


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
133238,PX,2.0,0.0,27.0,1.0,0.0,0.0,1.0,4.0
133239,PX,2.0,0.0,11.0,1.0,0.0,0.0,1.0,4.0
133240,PX,2.0,0.0,9.0,0.0,0.0,0.0,1.0,4.0
133241,PX,2.0,0.0,26.0,0.0,0.0,0.0,1.0,4.0
133242,PX,2.0,0.0,26.0,0.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
139080,PX,4.0,0.0,138.0,0.0,0.0,0.0,0.0,1.0
139081,PX,4.0,0.0,133.0,0.0,0.0,0.0,0.0,1.0
139082,PX,4.0,0.0,133.0,0.0,0.0,0.0,0.0,1.0
139083,PX,4.0,0.0,41.0,0.0,0.0,0.0,0.0,1.0


Domain: START  →  Number of rows: 32393


,domain,residue_name,IBS,residue_number,convhull_vertex,is_hydrophobic_protrusion,is_co_insertable,exposed,type
46690,START,2.0,0.0,79.0,1.0,0.0,0.0,1.0,4.0
46691,START,2.0,0.0,95.0,0.0,0.0,0.0,0.0,4.0
46692,START,2.0,1.0,129.0,0.0,0.0,0.0,0.0,4.0
46693,START,2.0,0.0,154.0,0.0,0.0,0.0,0.0,4.0
46694,START,2.0,0.0,15.0,1.0,0.0,0.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...
79078,START,4.0,0.0,207.0,0.0,0.0,0.0,0.0,1.0
79079,START,4.0,0.0,207.0,0.0,0.0,0.0,0.0,1.0
79080,START,4.0,0.0,207.0,0.0,0.0,0.0,0.0,1.0
79081,START,4.0,0.0,207.0,0.0,0.0,0.0,0.0,1.0


In [6]:
for domain_name, subset_df in groups.items():
    filename = f"{domain_name}_subset.csv"
    subset_df.to_csv(filename, index=False)


# ANNEXIN 

In [8]:
annexin_df = load_data("/home/user_stel/AISB/Project/dataset/ANNEXIN_subset.csv")

columns_to_remove = ['domain', 'residue_number']
X, y = separate_features_target(annexin_df, target='IBS', columns_to_remove=columns_to_remove)

In [ ]:
model = RandomForestClassifier(max_depth = 10, min_samples_leaf = 1, min_samples_split = 2, n_estimators = 200)

def train_final_model(model, X, y, save_path, scale=True):
        """
        Train model instance with hyperparameters.
        Save best model instance in provided path.
        """

        # optional global scaling
        if scale:
            scaler = StandardScaler()
            X = scaler.fit_transform(X)

        # fit on all available data
        model.fit(X, y)

        # persist
        folder = os.path.dirname(save_path)
        if folder:
            os.makedirs(folder, exist_ok=True)
        joblib.dump(model, save_path)
        print(f"Final model trained on all data and saved to {save_path}")

        return model

def evaluate_model(model, X, y, runs=30, test_size=0.2, scale=True, save_path=None):
        """
        Evaluate model instance. 
        """
        metrics = {
            'auc':   [],
            'prauc': [],
            'mcc':   []
        }

        best_auc = 0.0
        best_model = None

        for i in range(runs):
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)
            if scale:
                scaler=StandardScaler()
                X_train=scaler.fit_transform(X_train)
                model.fit(X_train, y_train)
                X_test=scaler.fit_transform(X_test)
                y_pred = model.predict(X_test)
             
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

            if hasattr(model, "predict_proba"):
                y_scores = model.predict_proba(X_test)[:, 1]
            else:
                y_scores = model.decision_function(X_test)

            cm = confusion_matrix(y_test, y_pred)
            cm_df = pd.DataFrame(
                cm,
                index=["Actual: 0", "Actual: 1"],
                columns=["Predicted: 0", "Predicted: 1"]
            )
            plt.figure(figsize=(5,4))
            sns.heatmap(cm_df, annot=True, fmt="d", cmap=cmap)
            plt.title(f"{model.__class__.__name__} Confusion Matrix")
            plt.ylabel("True label")
            plt.xlabel("Predicted label")
            plt.show()


            auc_val   = roc_auc_score(y_test, y_scores)
            prauc_val = average_precision_score(y_test, y_scores)
            mcc_val   = matthews_corrcoef(y_test, y_pred)
            
            metrics['auc'].append(auc_val)
            metrics['prauc'].append(prauc_val)
            metrics['mcc'].append(mcc_val)
            
            print(f"[Run {i+1}] AUC: {auc_val:.4f}, PR AUC: {prauc_val:.4f}, MCC: {mcc_val:.4f}")
            
            # Track the best model
            if auc_val > best_auc:
                best_auc   = auc_val
                best_model = np.copy.deepcopy(model)
                print(f"  ↳ New best model (AUC={best_auc:.4f})")
        
        results = {k: self.summarize(v) for k, v in metrics.items()}

        if save_path is not None and best_model is not None:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            joblib.dump(best_model, save_path)
            print(f"Best model (highest AUC: {best_auc:.4f}) saved to {save_path}")

        for metric, values in metrics.items():
            plt.figure(figsize=(8, 6))
            sns.boxplot(y=values)
            plt.title(f"{metric.upper()} Distribution")
            plt.ylabel(metric.upper())
            plt.show()

        return results